In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import json

# ---------------------- 1、定义本地工具函数（你要让AI调用的函数） ----------------------
def get_weather(city: str) -> str:
    """获取指定城市的天气
    Args:
        city: 城市名称
    """
    # 模拟接口，真实项目这里调用第三方天气API
    mock_data = {
        "武汉": "温度28℃，多云",
        "北京": "温度24℃，晴天",
        "上海": "温度30℃，小雨"
    }
    return mock_data.get(city, f"暂无【{city}】天气数据")


def calculator(a: float, b: float, op: str) -> str:
    """简单计算器，支持 + - * /
    Args:
        a: 数字1
        b: 数字2
        op: 运算符，可选 + - * /
    """
    if op == "+":
        res = a + b
    elif op == "-":
        res = a - b
    elif op == "*":
        res = a * b
    elif op == "/":
        res = a / b
    else:
        return "不支持该运算符"
    return f"计算结果 = {res}"


# 工具映射：工具名字 → 真实python函数
tool_map = {
    "get_weather": get_weather,
    "calculator": calculator
}

# 给大模型看的工具描述 schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询某个城市当前天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名，例如武汉"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "数学计算器，做加减乘除运算",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                    "op": {"type": "string", "description": "运算符 + - * /"}
                },
                "required": ["a", "b", "op"]
            }
        }
    }
]

# ---------------------- 2、初始化客户端 ----------------------
client = OpenAI(
    api_key="你的api-key",
    base_url="https://api.deepseek.com"   # 兼容OpenAI接口的地址，可换成minimax/openai
)


def simple_tool_call(user_query: str):
    messages = [{"role": "user", "content": user_query}]

    # 第一轮：请求大模型，看是否需要调用工具
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    ai_msg = resp.choices[0].message

    # --- 判断：是否触发工具调用 ---
    if not ai_msg.tool_calls:
        # 不需要工具，直接返回回答
        return ai_msg.content

    # ========== 需要执行工具调用，手写工具执行逻辑 ==========
    # 1）把AI返回的tool_call加入会话
    messages.append(ai_msg.model_dump())

    # 2）循环执行每一个工具
    for tool_call in ai_msg.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)

        # 拿到本地函数执行
        func = tool_map[func_name]
        tool_result = func(**func_args)

        # 3）把工具运行结果回写到messages，格式必须遵守openai协议
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result
        })

    # 第二轮：把工具结果丢回大模型，生成自然语言最终答案
    final_resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages
    )
    return final_resp.choices[0].message.content


# ---------------------- 测试 ----------------------
if __name__ == "__main__":
    print(simple_tool_call("武汉现在天气怎么样？"))
    print("-"*60)
    print(simple_tool_call("123 乘以 456等于多少"))
    print("-"*60)
    print(simple_tool_call("你好，随便聊聊"))
